# Lab 23 · Stream responses and count every token

Stream a chat completion and a Responses API reply through the gateway, see which chunks a streaming client receives
and when the usage arrives, then put a token limit on a streaming alias and prove that it counts streamed tokens
whether or not the client asks for usage. This notebook is the Python track of the [lab guide](README.md): it streams
with the same calls as [infra/stream.py](infra/stream.py), deploys the token-limit template from
[lab 06](../06-token-limits/infra/limited-alias.bicep), and runs the same burst as
[infra/stream-burst.sh](infra/stream-burst.sh).

The template doesn't touch the model or alias that your clients already call. It adds the alias
`gpt-6-luna-stream`, which targets the same backend model and carries its own token limit.

**Check the model → Stream a chat → Ask for usage → Stream the Responses API → Deploy the alias → Burst → Inspect 429 → Clean up**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Backend model `foundry-models/gpt-6-luna` | The model from notebook 02. The first streams call it through its `gpt-6-luna` alias. |
| Alias `gpt-6-luna-stream` | Targets the same model and carries a `tokenLimit` policy: 1,000 tokens per minute for each caller. |
| Runtime key `app-first-call` | The caller that streams and then runs out of tokens. |
| [../06-token-limits/infra/limited-alias.bicep](../06-token-limits/infra/limited-alias.bicep) | Declares an alias with a token limit. This notebook reuses it with `aliasName=gpt-6-luna-stream`. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![Streamed responses: the gateway checks limits before the stream, relays chunks as they arrive, and counts usage from every stream](../assets/diagrams/streaming-usage.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-6-luna` model.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q --upgrade azure-identity requests openai

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. The streaming alias targets
the backend model `gpt-6-luna` under the `foundry-models` provider that notebook 02 published.

In [ ]:
import sys
sys.path.insert(0, "../shared")   # helpers that every lab notebook shares
import utils

lab = utils.Lab("23-streaming-responses")
provider     = "foundry-models"     # model provider from notebook 02
model        = "gpt-6-luna"         # alias from notebook 02, and the backend model the streaming alias targets
stream_alias = f"{model}-stream"    # created and deleted by this notebook

### 1. Check the model and get a key

The cell confirms that notebook 02 published the model, then reuses the `app-first-call` key, or creates it.

In [ ]:
lab.require(f"{lab.workspace_id}/modelProviders/{provider}/models/{model}", "Run 02-first-model-call.ipynb first.")
print(f"✅ Model {provider}/{model} exists")

api_key, created_key = lab.api_key("app-first-call")
print(("✅ Created" if created_key else "♻️ Reusing") + f" key app-first-call: {api_key[:4]}…")

### 2. Stream a chat completion

With `stream=True`, the gateway answers `200` with `content-type: text/event-stream` and relays each chunk as the
model produces it. The `stream_chat` helper prints the text as it arrives and records the time to the first token.

Two kinds of chunks carry no text: the first chunk holds the prompt filter results, and the last one holds usage when
the client asks for it. Both have an empty `choices` array, so the helper skips them before it reads
`choices[0]`. This call doesn't ask for usage.

In [ ]:
import time
from openai import OpenAI

base_url = f"{lab.gateway_url}/default/models/openai/v1"
client   = OpenAI(base_url=base_url, api_key=api_key)
QUESTION = "Explain in two sentences why streaming makes chat apps feel faster."

def stream_chat(client, model, prompt=QUESTION, include_usage=False, show=True, **options):
    """Stream one chat completion. Return the number of chunks, the time to the first token, and the usage, if any."""
    started, first_token, usage, chunks = time.perf_counter(), None, None, 0
    extra = {"stream_options": {"include_usage": True}} if include_usage else {}
    stream = client.chat.completions.create(model=model, stream=True, **extra, **options,
                                            messages=[{"role": "user", "content": prompt}])
    for chunk in stream:
        chunks += 1
        usage = chunk.usage or usage
        if not chunk.choices:   # the prompt filter chunk and the usage chunk have no choices
            continue
        text = chunk.choices[0].delta.content
        if text:
            first_token = first_token or time.perf_counter() - started
            if show:
                print(text, end="", flush=True)
    return chunks, first_token, usage

chunks, first_token, usage = stream_chat(client, model)
print(f"\n\n📦 Chunks:      {chunks}")
print(f"⚡ First token: {first_token:.2f} s")
print(f"🧮 Usage chunk: {usage or 'none, the client did not ask for one'}")

### 3. Ask for usage at the end of the stream

Set `stream_options={"include_usage": True}` and the stream ends with one more chunk: an empty `choices` array and a
`usage` object, just before `data: [DONE]`. Clients that bill, log, or show token counts read it there.

In [ ]:
chunks, first_token, usage = stream_chat(client, model, include_usage=True)
print(f"\n\n📦 Chunks:      {chunks}")
print(f"🧮 Usage chunk: {usage.total_tokens} tokens (prompt {usage.prompt_tokens}, completion {usage.completion_tokens})")
assert usage and usage.total_tokens > 0

### 4. Stream the Responses API

The Responses API streams typed events instead of chunks. Text arrives in `response.output_text.delta` events, and
`response.completed` always carries the usage, so there's no option to set.

In [ ]:
from collections import Counter

events = Counter()
for event in client.responses.create(model=model, input="Write a one-sentence status update about streaming.", stream=True):
    events[event.type] += 1
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        done = event.response.usage

print("\n")
for name, count in events.items():
    print(f"{count:>4}  {name}")
print(f"\n🧮 Usage: {done.total_tokens} tokens (input {done.input_tokens}, output {done.output_tokens})")

### 5. Deploy a streaming alias with a token limit

Reuse the template from lab 06 with `aliasName` set to `gpt-6-luna-stream`. It creates the alias with a token limit
of 1,000 tokens per minute, counted per caller identity, so every runtime key gets its own counter.

In [ ]:
import json

lab.deploy("../06-token-limits/infra/limited-alias.bicep", gatewayName=lab.gateway_name, providerName=provider,
           modelName=model, aliasName=stream_alias, allowedTokens=1000, period="minute", counterKey=["identity"])

policies = lab.arm("GET", f"{lab.workspace_id}/aliases/{stream_alias}")["properties"]["policies"]
print(f"🏷️ Alias {stream_alias} policies:")
print(json.dumps(policies, indent=2))

### 6. Stream a burst without asking for usage

Stream requests of about 225 tokens each to the alias, without `include_usage`, so the client never receives a usage
chunk. The gateway still counts each stream's tokens, and after a few requests it answers `429`. The burst client sets
`max_retries=0`: the OpenAI SDK normally honors `Retry-After` and retries for you, which would hide the `429`.

A new alias can take a minute to apply, so the cell first waits until the alias answers, and repeats the burst if
no request is throttled.

In [ ]:
from openai import RateLimitError

PROMPT  = "Write a 150-word product update about an AI gateway that governs models and tools."
burster = OpenAI(base_url=base_url, api_key=api_key, max_retries=0)   # show the 429 instead of retrying it

def burst(include_usage=False, count=10):
    """Stream requests until the gateway answers 429. Return the RateLimitError, or None."""
    print(f"{'Request':<8} {'Status':<7} {'Retry-After':<12} {'Chunks':<7} Usage chunk")
    for i in range(1, count + 1):
        try:
            chunks, _, usage = stream_chat(burster, stream_alias, PROMPT, include_usage, show=False,
                                           max_completion_tokens=250, reasoning_effort="none")
            print(f"{i:<8} {200:<7} {'-':<12} {chunks:<7} {f'{usage.total_tokens} tokens' if usage else 'none'}")
        except RateLimitError as error:
            message = error.response.json().get("message", "")
            print(f"{i:<8} {429:<7} {error.response.headers.get('retry-after', '-'):<12} {'-':<7} {message}")
            return error
    return None

utils.retry(lambda: burster.chat.completions.create(   # wait until the new alias answers
    model=stream_alias, max_completion_tokens=16, messages=[{"role": "user", "content": "ping"}]), statuses=(401, 404, 429))

for attempt in range(3):
    throttled = burst()
    if throttled:
        break
    print("⏳ No 429 yet. The policy may still be applying; trying again in 20 s.")
    time.sleep(20)

assert throttled, "Expected a 429."
print("\n✅ The token limit counted the streams, although the client never received a usage chunk")

### 7. Inspect the throttled request

The gateway checks the limit before it calls the model, so a throttled streaming request never becomes a stream. It's
a plain JSON response with `Retry-After`, and a streaming client has to handle it before it parses any events. A
stream that has already started always finishes: the limit applies to the next request.

In [ ]:
headers = throttled.response.headers
print(f"📨 HTTP {throttled.status_code}")
for name in ("content-type", "retry-after"):
    print(f"   {name}: {headers.get(name)}")
print(f"   body: {throttled.response.text}")
assert headers["content-type"].startswith("application/json")

### 8. Stream a burst with usage

Wait a minute for a fresh counter, then run the same burst with `include_usage`. The client now sees each stream's
tokens, and the gateway throttles at about the same request as before: asking for usage changes what the client
sees, not what the gateway counts.

In [ ]:
print("⏳ Waiting 60 s for a fresh counter...")
time.sleep(60)
assert burst(include_usage=True), "Expected a 429."

### 9. Clean up resources

The streaming alias throttles anyone who calls it, so the cell deletes it. The model and its `gpt-6-luna` alias were
never changed. The cell deletes `app-first-call` only if this notebook created it.

To find the streamed calls in **Monitoring**, configure Application Insights first (lab 13).

In [ ]:
lab.delete(f"{lab.workspace_id}/aliases/{stream_alias}")
if created_key:
    lab.delete(f"{lab.gateway_id}/apiKeys/app-first-call")

## Troubleshooting

- **`IndexError` on `choices[0]`**: the client read a chunk with an empty `choices` array, the first chunk (prompt
  filter results) or the usage chunk. Skip chunks without choices, as `stream_chat` does.
- **No usage chunk**: the client didn't set `stream_options={"include_usage": True}`. The gateway counts the tokens
  either way; set it when the client needs the numbers itself.
- **`400` that mentions `stream_options`**: `stream_options` is only allowed when `stream` is `True`.
- **No `429` after several bursts**: check that the request `model` is the streaming alias, and that
  `lab.arm("GET", f"{lab.workspace_id}/aliases/{stream_alias}")["properties"]["policies"]` lists the token limit.
- **`404` with `model_not_found` right after step 5**: a new alias takes up to a minute to apply. The burst cell waits
  for it; rerun the cell if it timed out.
- **The OpenAI SDK waits instead of failing**: it honors `Retry-After` and retries. Set `max_retries=0` to see the
  `429`.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-23-streaming-responses).

## Next steps

- Cap spend in dollars per caller in [Enforce cost budgets with per-caller overrides](../07-cost-budgets/README.md).
- See every streamed call's tokens and cost in [Monitor usage, cost, and traces](../13-monitoring/README.md).
- Stream from agent frameworks in [Connect agent frameworks to models and tools](../15-agent-frameworks/README.md).
- Docs: [Releases: September 24, 2026](https://ai.gateway.azure.com/docs/releases/2026-09-24).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/23-streaming-responses/](https://azure.github.io/ai-gateway/labs/23-streaming-responses/).